# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [1]:
import numpy as np
from collections import defaultdict

class WarehouseMDP:
    def __init__(self):
        self.height, self.width = 5, 6
        self.start = (0, 0)
        self.walls = {(0, 3), (1, 1), (2, 4), (4, 2)}
        self.slippery_states = {(1, 2), (2, 1), (3, 3)}
        self.terminal_states = {(0, 5): 10.0, (2, 2): 2.0, (3, 5): -10.0}
        self.danger_states = {(1, 4): -3.0, (4, 1): -3.0}
        self.living_reward = -1.0
        self.gamma = 0.9
        self.slippery_forward_prob = 0.6
        self.actions = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    def is_valid_state(self, state):
        r, c = state
        return 0 <= r < self.height and 0 <= c < self.width and state not in self.walls

    def states(self):
        return [(r, c) for r in range(self.height) for c in range(self.width)
                if self.is_valid_state((r, c))]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        if state in self.terminal_states:
            return self.terminal_states[state]
        return self.danger_states.get(state, self.living_reward)

    def get_transition_probs(self, state, action):
        if self.is_terminal(state):
            return [(state, 1.0)]
        dr, dc = action
        left, right = (-dc, dr), (dc, -dr)
        forward = self.slippery_forward_prob if state in self.slippery_states else 0.9
        side = (1.0 - forward) / 2
        outcomes = defaultdict(float)
        for (vr, vc), p in [(action, forward), (left, side), (right, side)]:
            nxt = (state[0] + vr, state[1] + vc)
            outcomes[nxt if self.is_valid_state(nxt) else state] += p
        return list(outcomes.items())



### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [2]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [3]:
def expected_next_value(grid, state, action, V):
    return sum(p * V[next_state] for next_state, p in grid.get_transition_probs(state, action))

def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    S = grid.states()
    V = {s: grid.get_reward(s) if grid.is_terminal(s) else 0.0 for s in S}
    for iteration in range(1, max_iter + 1):
        new_V = V.copy()
        for s in S:
            if not grid.is_terminal(s):
                new_V[s] = grid.get_reward(s) + grid.gamma * max(
                    expected_next_value(grid, s, a, V) for a in grid.actions)
        delta = max(abs(new_V[s] - V[s]) for s in S)
        V = new_V
        if delta < threshold:
            return V, iteration
    raise RuntimeError("Value Iteration no convergió")

def extract_policy(grid, V):
    return {s: max(grid.actions, key=lambda a: expected_next_value(grid, s, a, V))
            for s in grid.states() if not grid.is_terminal(s)}



## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [4]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    S = grid.states()
    V = {s: grid.get_reward(s) if grid.is_terminal(s) else 0.0 for s in S}
    for _ in range(max_iter):
        new_V = V.copy()
        for s in policy:
            new_V[s] = grid.get_reward(s) + grid.gamma * expected_next_value(grid, s, policy[s], V)
        delta = max(abs(new_V[s] - V[s]) for s in S)
        V = new_V
        if delta < threshold:
            return V
    raise RuntimeError("Policy Evaluation no convergió")

def policy_improvement(grid, V):
    return extract_policy(grid, V)

def policy_iteration(grid, threshold=1e-4, max_iter=100):
    policy = {s: grid.actions[0] for s in grid.states() if not grid.is_terminal(s)}
    history = []
    for iteration in range(1, max_iter + 1):
        V = policy_evaluation(grid, policy, threshold)
        new_policy = policy_improvement(grid, V)
        changes = sum(policy[s] != new_policy[s] for s in policy)
        history.append({"iteracion": iteration, "cambios": changes})
        if changes == 0:
            return new_policy, V, history
        policy = new_policy
    raise RuntimeError("Policy Iteration no convergió")



## Parte 4 — Visualización y comparación


In [5]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [6]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert max(abs(V_vi[s] - V_pi[s]) for s in grid.states()) < 1e-3
print("Coincidencias de política:", sum(pi_vi[s] == pi_pi[s] for s in pi_vi), "/", len(pi_vi))
print("\n✓ Ambos algoritmos obtuvieron valores óptimos equivalentes.")


=== VALUE ITERATION ===
Iteraciones: 19

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== POLICY ITERATION ===
Historia: [{'iteracion': 1, 'cambios': 17}, {'iteracion': 2, 'cambios': 5}, {'iteracion': 3, 'cambios': 1}, {'iteracion': 4, 'cambios': 0}]

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Polít


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
2. ¿Por qué una recompensa menor podría ser óptima?
3. ¿En qué estados el piso resbaloso cambia la decisión?
4. ¿Qué papel cumple el costo por paso `-1`?
5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.


### Código para realizar los experimentos

```python
import copy
for nombre, cambios in [
    ("A: costo -0.1", {"living_reward": -0.1}),
    ("B: resbaloso 0.40", {"slippery_forward_prob": 0.40}),
    ("C: gamma 0.99", {"gamma": 0.99}),
]:
    escenario = copy.deepcopy(grid)
    for atributo, valor in cambios.items():
        setattr(escenario, atributo, valor)
    valores, _ = value_iteration(escenario)
    politica = extract_policy(escenario, valores)
    print(nombre, "inicio:", ARROWS[politica[escenario.start]])
    print_policy(escenario, politica)

for costo in np.arange(-1.0, 0.51, 0.05):
    escenario = copy.deepcopy(grid)
    escenario.living_reward = float(costo)
    valores, _ = value_iteration(escenario)
    politica = extract_policy(escenario, valores)
    print(f"costo={costo:+.2f}, V(start)={valores[escenario.start]:+.3f}, acción={ARROWS[politica[escenario.start]]}")
```


### Respuestas de interpretación

1. **¿A dónde va desde el inicio?** Con los valores originales, el robot prefiere ir a la estación de carga (+2). Si todo sale como espera, pasa por `(0,1)`, `(0,2)` y `(1,2)` antes de llegar a `(2,2)`. Como hay movimientos aleatorios, no siempre va a seguir exactamente ese camino.

2. **¿Por qué escoge +2 y no +10?** Al principio pensé que escogería la entrega porque da más puntos. Pero para llegar allá tiene que recorrer más casillas, gastar puntos en cada movimiento y arriesgarse a caer en una zona de peligro. Entonces, con estas condiciones, le sale mejor terminar antes con +2.

3. **¿Qué pasa con el piso resbaloso?** En esas casillas el robot tiene menos posibilidades de moverse justo hacia donde quiere. Eso afecta el resultado, aunque no necesariamente cambia la dirección que escoge. Por ejemplo, al bajar la probabilidad de avanzar a 0.40, las flechas quedaron iguales, pero el valor desde el inicio bajó de aproximadamente -2.575 a -2.706.

4. **¿Para qué sirve el costo de -1 por paso?** Hace que el robot no dé vueltas innecesarias. Cada movimiento le cuesta, así que una ruta larga puede terminar siendo peor, incluso si la recompensa final es mayor. Cuando bajé ese costo a -0.1, ya empezó a convenirle ir por la entrega.

5. **¿Por qué las probabilidades de transición no son iguales en todo el mapa?** Porque no todas las casillas funcionan igual. En una normal, la probabilidad de avanzar como se pidió es 0.90; en una resbalosa es 0.60. Además, si intenta pasar por una pared o salirse del mapa, se queda donde estaba. Por eso hay que calcular las transiciones según la casilla y sus alrededores.

### Lo que pasó en los experimentos

- **A: costo por paso de -0.1.** Esperaba que el robot se animara a buscar la entrega porque moverse ya no le costaba tanto. Eso fue lo que pasó: en `(1,2)` cambió de bajar a seguir a la derecha. El valor desde START fue aproximadamente 1.649.
- **B: piso más resbaloso (0.40).** Pensé que podía cambiar de ruta para evitar algunas casillas. Al ejecutar, vi que las direcciones no cambiaron, pero sí empeoró el valor inicial, que quedó cerca de -2.706. O sea, la ruta sigue siendo la mejor entre las opciones, aunque ahora sea más difícil recorrerla.
- **C: gamma de 0.99.** Al darle más importancia a lo que puede ganar después, esperaba que la entrega +10 fuera más atractiva. En `(1,2)` efectivamente cambió la decisión hacia la derecha, y el valor inicial quedó cerca de -1.456.


In [7]:
import copy
for nombre, cambios in [
    ("A: costo -0.1", {"living_reward": -0.1}),
    ("B: resbaloso 0.40", {"slippery_forward_prob": 0.40}),
    ("C: gamma 0.99", {"gamma": 0.99}),
]:
    escenario = copy.deepcopy(grid)
    for atributo, valor in cambios.items():
        setattr(escenario, atributo, valor)
    valores, _ = value_iteration(escenario)
    politica = extract_policy(escenario, valores)
    print("\n", nombre, "V(start):", round(valores[escenario.start], 3))
    print_policy(escenario, politica)

print("\nBarrido de costo por paso")
for costo in np.arange(-1.0, 0.51, 0.05):
    escenario = copy.deepcopy(grid)
    escenario.living_reward = float(costo)
    valores, _ = value_iteration(escenario)
    politica = extract_policy(escenario, valores)
    print(f"costo={costo:+.2f}, V(start)={valores[escenario.start]:+.3f}, acción={ARROWS[politica[escenario.start]]}")



 A: costo -0.1 V(start): 1.649
 →  |  →  |  ↓  |  #  |  →  | +10
 ↑  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  →  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

 B: resbaloso 0.40 V(start): -2.706
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

 C: gamma 0.99 V(start): -1.456
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

Barrido de costo por paso
costo=-1.00, V(start)=-2.575, acción=→
costo=-0.95, V(start)=-2.369, acción=→
costo=-0.90, V(start)=-2.163, acción=→
costo=-0.85, V(start)=-1.957, acción=→
costo=-0.80, V(start)=-1.751, acción=→
costo=-0.75, V(start)=-1.511, acción=→
costo=-0.70, V(start)=-1.269, acción=→
costo=-0.65, V(start)=-1.028, acción=→
costo=-0.60, V(start)=-0.786, acción=→
costo=-0.55, 

costo=+0.30, V(start)=+3.629, acción=→
costo=+0.35, V(start)=+3.982, acción=→
costo=+0.40, V(start)=+4.336, acción=→
costo=+0.45, V(start)=+4.689, acción=→


costo=+0.50, V(start)=+5.043, acción=→


### Bonus — ¿Cuándo cambia de destino?

Para encontrar el cambio probé varios valores del costo por paso. Al principio miré la flecha de START, pero me di cuenta de que no servía para distinguir las dos rutas: en ambos casos el robot empieza moviéndose hacia la derecha.

Entonces revisé la casilla `(1,2)`, donde sí se separan los caminos. Con un costo de -0.80, el robot baja hacia la carga +2; con -0.775, sigue a la derecha hacia la entrega +10.

Después afiné el cálculo con una búsqueda binaria y encontré que el cambio ocurre cerca de **-0.79834**. Si el costo es más negativo, prefiere la carga; si es menos negativo, empieza a convenir la entrega. Esto tiene sentido porque, cuando moverse cuesta menos, vale más la pena intentar llegar a la recompensa grande.

Este resultado depende de las probabilidades y del valor de gamma que usamos. Además, como el movimiento no es totalmente seguro, escoger una ruta no significa que siempre vaya a llegar a ese destino. Tampoco se puede asumir que el mismo comportamiento se mantiene si damos recompensas positivas por cada paso, porque ahí podría convenirle seguir moviéndose sin terminar.


In [8]:
# Búsqueda binaria del umbral en el cruce (1,2)
cruce = (1, 2)
lim_inf, lim_sup = -0.80, -0.775
for _ in range(22):
    medio = (lim_inf + lim_sup) / 2
    escenario = copy.deepcopy(grid)
    escenario.living_reward = medio
    valores, _ = value_iteration(escenario, threshold=1e-9)
    accion = extract_policy(escenario, valores)[cruce]
    if accion == (1, 0):  # hacia abajo: carga
        lim_inf = medio
    else:                 # hacia la derecha: entrega
        lim_sup = medio
print(f"Umbral aproximado: {(lim_inf + lim_sup)/2:.6f}")
print("Bajo el umbral: ↓ hacia carga; sobre el umbral: → hacia entrega")

Umbral aproximado: -0.798341
Bajo el umbral: ↓ hacia carga; sobre el umbral: → hacia entrega
